# Build a Transformer, one step at a time

Sixteen small steps from raw text to a working GPT. Every step adds **one** idea, checks it against a reference implementation, and shows you the actual numbers.

The running example is deliberately tiny — 6 tokens, 4 dimensions — so every tensor fits on screen and you can read it. Nothing is hidden behind a shape you cannot print. We scale up only at the very end, once the mechanism is no longer mysterious.

**Prerequisites:** comfort with matrix multiplication. That is genuinely it.

---
## 0. Setup

`learnai.env.setup` seeds Python, NumPy and torch, picks the best device, and
prints what it found. Reproducibility first: every number in this notebook should
be the same on your machine as on mine.

In [ ]:
import math
import os

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from learnai.env import setup
from learnai.viz import use_style

ctx = setup(seed=0)
FAST = os.getenv("LEARNAI_FAST") == "1"  # tiny CI smoke run; normal lessons leave this unset
print(f"mode     {'FAST smoke test' if FAST else 'full lesson'}")
use_style()          # the house matplotlib theme, shared with the interactive pages
torch.set_printoptions(precision=3, sci_mode=False, linewidth=120)

---
## 1. The problem, stated concretely

Here is a sentence. We want a model where the representation of `"sat"` can
depend on the fact that `"cat"` appeared earlier.

A plain MLP cannot do this: it sees a fixed-size input and has no notion of "the
other tokens". An RNN can, but only by passing information through a serial chain
— token 6 reaches token 1 through five hops, and the gradient decays at each one.

Attention's answer: let **every position read from every other position directly**,
in one step, with the strength of each read decided by the content.

In [ ]:
sentence = "the cat sat on the mat"
words = sentence.split()  # a deliberately simple word split; production GPTs use subword tokens
T = len(words)          # sequence length
d_model = 4             # tiny on purpose: every tensor below is printable

print(f"{T} tokens, {d_model} dimensions each")
print(words)

---
## 2. Step 1 — Tokenization: text becomes integer IDs

Neural networks do not consume strings. A **tokenizer** first splits text into
symbols from a fixed vocabulary and maps each symbol to an integer. Here we use
whole words so the mapping fits on screen. Real GPT tokenizers use subwords or
bytes so unseen words can still be represented.

The IDs are merely addresses: `cat=1` does **not** mean cat is numerically closer
to `sat=2` than to another token. Decode the IDs immediately as a round-trip
check; a silent tokenizer mismatch corrupts every downstream example.

In [ ]:
vocab = ["the", "cat", "sat", "on", "mat"]
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
ids = torch.tensor([stoi[w] for w in words])
decoded = " ".join(itos[int(i)] for i in ids)

print("text:     ", sentence)
print("tokens:   ", words)
print("token ids:", ids.tolist())
print("decoded:  ", decoded)
assert decoded == sentence
print("encode -> decode round-trip ✓")

---
## 3. Step 2 — Embedding: IDs become vectors

An integer ID still has no useful geometry. The first learned layer is a lookup
table mapping each ID to a vector. `nn.Embedding` is exactly a matrix of shape
`(vocab_size, d_model)` whose forward pass selects rows. A one-hot matrix multiply
would be mathematically identical, but would materialise a huge sparse tensor.

The vectors below are set **by hand** so later attention patterns mean something
you can verify by eye. In a real model they start random and are learned.

In [ ]:
# Hand-built vectors. Dimensions mean, roughly:
#   0: "is a determiner"   1: "is a noun"   2: "is a verb"   3: "is animate"
table = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],   # the  — determiner
    [0.0, 1.0, 0.0, 1.0],   # cat  — noun, animate
    [0.0, 0.0, 1.0, 0.0],   # sat  — verb
    [0.0, 0.0, 1.0, 0.0],   # on   — (treated as a relation; verb-ish column)
    [0.0, 1.0, 0.0, 0.0],   # mat  — noun, inanimate
])
emb = nn.Embedding.from_pretrained(table, freeze=True)

X = emb(ids)          # (T, d_model)
print("\nX  (T, d_model) =", tuple(X.shape))
for w, row in zip(words, X, strict=True):
    print(f"  {w:5s} {row.tolist()}")

**Takeaway**

- Tokenization is a reversible contract between text and integer IDs; IDs themselves carry no meaning.
- `nn.Embedding` is a lookup table of shape `(vocab_size, d_model)`, nothing more.
- Row *i* is the model's entire prior knowledge about token *i* before any context is applied.
- Everything downstream operates on these vectors — the integers are gone after this line.

---
## 4. Step 3 — One token looks at another: the dot product

How much should `"sat"` care about `"cat"`? We need a **scalar** measuring how
compatible two vectors are. The dot product is the obvious candidate: it is large
and positive when two vectors point the same way, zero when they are orthogonal,
negative when they oppose.

Compute it by hand for one pair, then convince yourself the matrix version does
the same thing.

In [ ]:
i, j = 2, 1                      # query = "sat", key = "cat"
manual = sum(X[i][k] * X[j][k] for k in range(d_model))
print(f'{words[i]!r} · {words[j]!r} = {manual:.3f}   (by hand, term by term)')
print(f'{words[i]!r} · {words[j]!r} = {torch.dot(X[i], X[j]):.3f}   (torch.dot)')

# "cat" vs "cat" — a vector's dot product with itself is its squared length
print(f'\n{words[1]!r} · {words[1]!r} = {torch.dot(X[1], X[1]):.3f}  <- self-similarity is highest')

Note what just happened: `"sat"` and `"cat"` score **0.0** because our hand-built
vectors are orthogonal — a verb and a noun share no active dimension.

That is a problem. We *want* the verb to attend to its subject. With raw
embeddings, attention can only match tokens that are already similar, which is
almost never what you want. Step 8 fixes this with learned projections; hold the
thought.

---
## 5. Step 4 — All pairs at once

Doing that dot product for every `(query, key)` pair is one matrix multiply.
`X @ X.T` has shape `(T, T)`, and entry `[i, j]` is exactly `X[i] · X[j]`.

This is the single most important shape in the whole architecture. Every
`(T, T)` matrix you will ever see in a Transformer paper is this thing.

In [ ]:
scores = X @ X.T                # (T, T)
print("scores", tuple(scores.shape))
print(scores)

# verify against the by-hand value from the previous cell
assert torch.allclose(scores[2, 1], torch.dot(X[2], X[1]))
print("\nmatches the hand computation ✓")

In [ ]:
from learnai.viz import attention_heatmap

fig, ax = plt.subplots(figsize=(4.5, 4))
attention_heatmap(scores, tokens=words, ax=ax, title="raw scores  X @ X.T", show_values=True)
plt.show()

Read the picture: bright where two tokens share features. Both `"the"`s light up
against each other, both nouns against each other. It is a similarity matrix —
and notice it is **symmetric**, because `a·b == b·a`.

Real attention is *not* symmetric: "the" attending to "cat" is a different
question from "cat" attending to "the". That asymmetry arrives in step 6.

---
## 6. Step 5 — Scale by √d, and why it is not optional

Dot products over `d` dimensions have variance proportional to `d`. So as the
model gets wider, scores get larger — and softmax of large scores saturates into
a one-hot spike with essentially zero gradient.

Do not take this on faith. Measure it.

In [ ]:
def mean_entropy(d, scale):
    """Average entropy of attention rows for random Q, K of width d."""
    g = torch.Generator().manual_seed(0)
    q = torch.randn(16, d, generator=g)
    k = torch.randn(16, d, generator=g)
    w = torch.softmax(q @ k.T * scale, dim=-1)
    return float(-(w * w.clamp_min(1e-12).log()).sum(-1).mean())

dims = [4, 16, 64, 256, 1024]
print(f"{'d':>6} {'unscaled':>10} {'/ sqrt(d)':>10}   (max possible = %.2f)" % math.log(16))
for d in dims:
    print(f"{d:6d} {mean_entropy(d, 1.0):10.3f} {mean_entropy(d, d ** -0.5):10.3f}")

Unscaled, entropy falls toward 0 — by `d=1024` the softmax has collapsed onto a
single position and `∂loss/∂score ≈ 0` everywhere, so the head cannot learn.
Scaled, entropy stays flat at every width.

That is the whole content of the word *scaled* in "scaled dot-product attention".

In [ ]:
scaled_scores = scores / math.sqrt(d_model)
print(f"divided by sqrt({d_model}) = {math.sqrt(d_model):.3f}")
print(scaled_scores)

---
## 7. Step 6 — Softmax turns scores into weights

Scores are arbitrary real numbers. To use them as mixing weights they must be
non-negative and sum to 1 — that is, a probability distribution over "where to
look".

`softmax(x)ᵢ = exp(xᵢ) / Σⱼ exp(xⱼ)`, applied along the **last** dimension so each
*row* becomes a distribution.

In [ ]:
from learnai.nn.functional import softmax as learnai_softmax

weights = torch.softmax(scaled_scores, dim=-1)     # (T, T)
print(weights)
print("\nrow sums:", weights.sum(dim=-1))

# what learnai.nn.functional.softmax does, written out (max-subtraction for stability)
assert torch.allclose(weights, learnai_softmax(scaled_scores), atol=1e-6)
print("matches learnai.nn.functional.softmax ✓")

Row `i` now answers: *"when computing the new representation of token i, what
fraction of it should come from each token?"*

One subtlety worth internalising: softmax is applied **per row**, so the columns
do **not** sum to 1. A token can be attended to heavily by everyone, or by no one.

---
## 8. Step 7 — Mix the values. Attention is now complete.

Multiply the weights by the vectors themselves and each token walks away with a
weighted average of the sequence.

That is it. `softmax(XXᵀ/√d) X` is attention. Everything else in a Transformer is
either making this learnable, running several in parallel, or stacking it.

In [ ]:
out = weights @ X               # (T, T) @ (T, d) -> (T, d)
print("out", tuple(out.shape))

# Check one row by hand: out[2] must equal sum_j weights[2,j] * X[j]
manual = sum(weights[2, j] * X[j] for j in range(T))
assert torch.allclose(out[2], manual, atol=1e-6)
print(f"\nout[2] ({words[2]!r}) = {out[2].tolist()}")
print(f"hand-computed          = {[round(v, 3) for v in manual.tolist()]}  ✓")

In [ ]:
# The same thing, from the library. Note the (B, H, T, D) shape convention:
# real attention always carries batch and head dimensions.
from learnai.nn.attention import scaled_dot_product_attention

q = X.unsqueeze(0).unsqueeze(0)        # (1, 1, T, d)
o_ref, w_ref = scaled_dot_product_attention(q, q, q)
assert torch.allclose(o_ref[0, 0], out, atol=1e-6)
assert torch.allclose(w_ref[0, 0], weights, atol=1e-6)
print("matches learnai.nn.attention.scaled_dot_product_attention ✓")

**Takeaway**

- Attention = score every pair, softmax the scores, average the values.
- The `(T, T)` weight matrix is the whole story — everything else is plumbing.
- It costs O(T²) memory and compute. That single fact drives the entire long-context research literature.

---
## 9. Step 8 — Q, K, V: make it learnable and asymmetric

So far a token plays all three roles with the same vector, which forces two
limitations we already noticed:

1. **Symmetry** — `score(i,j) == score(j,i)`, so "who modifies whom" is inexpressible.
2. **Content-only matching** — a token can only attend to tokens it already resembles.

The fix is three learned projections. Each token emits:

| | meaning | shape |
|---|---|---|
| **query** `q = x Wq` | what I am looking for | `(T, d)` |
| **key** `k = x Wk` | what I offer to others | `(T, d)` |
| **value** `v = x Wv` | what I actually hand over | `(T, d)` |

Now `score(i,j) = (xᵢWq)·(xⱼWk)` — no longer symmetric, and free to match on
*anything* the projections learn to extract.

To show it concretely, here are hand-built projections implementing the rule
*"verbs look for animate nouns"*.

In [ ]:
# dims: 0 = determiner, 1 = noun, 2 = verb, 3 = animate
# Wq maps "I am a verb" (dim 2) -> "I am looking for animate" (dim 3)
Wq = torch.zeros(d_model, d_model)
Wq[2, 3] = 1.0
# Wk is the identity: a token advertises its features as-is
Wk = torch.eye(d_model)
# Wv passes the token's own content along
Wv = torch.eye(d_model)

Q, K, V = X @ Wq, X @ Wk, X @ Wv

qkv_scores = Q @ K.T / math.sqrt(d_model)
qkv_weights = torch.softmax(qkv_scores, dim=-1)
qkv_out = qkv_weights @ V

# Same computation through the reusable reference, now with batch/head axes.
qkv_out_ref, qkv_weights_ref = scaled_dot_product_attention(
    Q[None, None], K[None, None], V[None, None]
)
assert torch.allclose(qkv_weights, qkv_weights_ref[0, 0], atol=1e-6)
assert torch.allclose(qkv_out, qkv_out_ref[0, 0], atol=1e-6)

print("attention row for 'sat' (a verb):")
for w, p in zip(words, qkv_weights[2], strict=True):
    bar = "#" * int(p * 40)
    print(f"  {w:5s} {p:.3f} {bar}")

`"sat"` now puts most of its weight on `"cat"` — the only animate token — even
though `sat · cat` was **0** back in step 3. The projection created a match that
did not exist in the raw embeddings.

And the matrix is no longer symmetric:

In [ ]:
print(f"score(sat -> cat) = {qkv_scores[2, 1]:.3f}")
print(f"score(cat -> sat) = {qkv_scores[1, 2]:.3f}   <- different, as it should be")
print(f"\nsymmetric? {torch.allclose(qkv_scores, qkv_scores.T)}")

---
## 10. Step 9 — The causal mask

For language modelling we predict token *t+1* from tokens *0..t*. If position 2
can see position 5, the model just reads the answer off the input and learns
nothing. So we forbid looking forward.

The implementation is one line — set forbidden scores to `-inf` **before** the
softmax, so `exp(-inf) = 0` and they receive exactly zero weight.

In [ ]:
from learnai.nn.attention import causal_mask

mask = causal_mask(T)[0, 0]          # (T, T) bool, True = allowed
print("mask (True = may attend):")
print(mask.int())

masked_scores = qkv_scores.masked_fill(~mask, float("-inf"))
causal_weights = torch.softmax(masked_scores, dim=-1)
causal_out = causal_weights @ V

causal_out_ref, causal_weights_ref = scaled_dot_product_attention(
    Q[None, None], K[None, None], V[None, None], mask=causal_mask(T)
)
assert torch.allclose(causal_weights, causal_weights_ref[0, 0], atol=1e-6)
assert torch.allclose(causal_out, causal_out_ref[0, 0], atol=1e-6)

print("\ncausal attention weights:")
print(causal_weights)
print("\nrow sums still 1:", causal_weights.sum(-1))
print("upper triangle exactly zero:", bool((causal_weights.triu(1) == 0).all()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
attention_heatmap(qkv_weights, tokens=words, ax=axes[0], title="bidirectional (BERT-style)",
                  show_values=True, cbar=False)
attention_heatmap(causal_weights, tokens=words, ax=axes[1], title="causal (GPT-style)",
                  show_values=True, cbar=False)
plt.tight_layout()
plt.show()

Those two pictures capture the core self-attention difference between a
bidirectional encoder such as BERT and a decoder-only GPT: same mechanism, one
mask changed. Their training objectives differ too (masked-token vs next-token).

Note row 0: token `"the"` can only see itself, so its weight is forced to 1.0 no
matter what the scores were. It keeps its own content and position, but receives
no information from any *other* token.

> **Your turn.** Build the mask for a **sliding window** of size 3 — each token may see itself and the two tokens before it. This is what Mistral and Longformer use to make attention cost O(T·w) instead of O(T²).
>
> <details><summary>hint</summary>
>
> `torch.ones(T, T).tril()` gives you causal; subtract off a `tril(-3)` to cut the tail.
>
> </details>

In [ ]:
window = 3
# build a (T, T) bool mask, then check it against the causal one

---
## 11. Step 10 — Multiple heads

One softmax produces **one** distribution per token — it can only average one
thing at a time. But "sat" wants to track its subject *and* its object *and* the
clause boundary.

Multi-head attention splits `d_model` into `H` chunks of size `d_model/H` and runs
an independent attention in each. Cost is unchanged: the head dimension shrinks to
compensate.

The tensor gymnastics are the only fiddly part, so let us do them slowly.

In [ ]:
d_model_big, n_heads = 8, 2
d_head = d_model_big // n_heads
B = 1

x = torch.randn(B, T, d_model_big)
print(f"x                          {tuple(x.shape)}   (B, T, d_model)")

qkv = nn.Linear(d_model_big, 3 * d_model_big)(x)
print(f"after fused qkv projection {tuple(qkv.shape)}   (B, T, 3*d_model)")

qh, kh, vh = qkv.chunk(3, dim=-1)
print(f"after chunk(3)             {tuple(qh.shape)}   x3")

# split the feature axis into heads, then move heads next to batch so that the
# matmul treats each head as a fully independent problem
qh = qh.view(B, T, n_heads, d_head).transpose(1, 2)
kh = kh.view(B, T, n_heads, d_head).transpose(1, 2)
vh = vh.view(B, T, n_heads, d_head).transpose(1, 2)
print(f"after view + transpose     {tuple(qh.shape)}   (B, H, T, d_head)")

oh, attn_h = scaled_dot_product_attention(qh, kh, vh, mask=causal_mask(T))
print(f"attention output           {tuple(oh.shape)}")
print(f"attention weights          {tuple(attn_h.shape)}   <- one (T,T) map PER HEAD")

# undo the split: contiguous() is required because transpose only changed strides
merged = oh.transpose(1, 2).contiguous().view(B, T, d_model_big)
print(f"after merging heads back   {tuple(merged.shape)}   (B, T, d_model)")

The `view → transpose → attention → transpose → view` sandwich is worth
memorising; you will read it in every Transformer codebase you ever open.

`contiguous()` matters: `transpose` only edits stride metadata, it does not move
memory, and `view` refuses to work on a non-contiguous tensor.

In [ ]:
from learnai.nn.attention import MultiHeadAttention

mha = MultiHeadAttention(d_model=8, n_heads=2, causal=True)
mha.eval()
y, a = mha(x, return_attn=True)

# Open the module and repeat its forward pass by hand with the SAME parameters.
q_m, k_m, v_m = mha.qkv_proj(x).chunk(3, dim=-1)

def split_heads(z):
    return z.view(B, T, n_heads, d_head).transpose(1, 2)

o_m, a_m = scaled_dot_product_attention(
    split_heads(q_m), split_heads(k_m), split_heads(v_m), mask=causal_mask(T)
)
merged_m = o_m.transpose(1, 2).contiguous().view(B, T, d_model_big)
y_m = mha.out_proj(merged_m)
assert torch.allclose(a, a_m, atol=1e-6)
assert torch.allclose(y, y_m, atol=1e-6)
print(f"MultiHeadAttention: {tuple(x.shape)} -> {tuple(y.shape)}, attn {tuple(a.shape)}")
print(f"parameters: {sum(p.numel() for p in mha.parameters()):,}")
print("\nrows sum to 1:", torch.allclose(a.sum(-1), torch.ones(1, 2, T)))

---
## 12. Step 11 — The feed-forward network

Attention moves information **between** positions. It does no per-token
computation to speak of — before the final output projection, each head produces
a convex combination of value vectors.

The MLP is where each token thinks privately. Same weights at every position, zero
mixing:

`FFN(x) = W₂ · GELU(W₁x)`, with `W₁` widening to `4 · d_model`.

That factor of 4 has survived unchanged from 2017 to today, and **two thirds of a
Transformer's parameters live here** — not in attention.

In [ ]:
from learnai.nn.blocks import FeedForward

ffn = FeedForward(d_model=8)
ffn.eval()
print(f"FFN: 8 -> {ffn.fc_in.out_features} -> 8")
print(f"FFN parameters:  {sum(p.numel() for p in ffn.parameters()):,}")
print(f"MHA parameters:  {sum(p.numel() for p in mha.parameters()):,}")

# position-independence: run one token alone, and inside a batch — identical result
single = ffn(x[:, 2:3, :])
in_seq = ffn(x)[:, 2:3, :]
manual_ffn = ffn.fc_out(torch.nn.functional.gelu(ffn.fc_in(x), approximate="tanh"))
assert torch.allclose(ffn(x), manual_ffn, atol=1e-6)
print(f"\nsame result for a token alone vs in context: {torch.allclose(single, in_seq, atol=1e-6)}")
print("  -> the MLP genuinely cannot see other positions")

---
## 13. Step 12 — Residuals and LayerNorm: the block

Two sub-layers, wrapped:

```
x = x + Attention(LayerNorm(x))
x = x + MLP(LayerNorm(x))
```

**The residual `x + ...` is the most important part of this notebook.** Think of
the residual stream as a shared workspace of shape `(B, T, d_model)`. Each
sub-layer *reads* a normalised copy and *adds* its contribution. Nothing is ever
overwritten. This is why gradients reach layer 1 of a 100-layer model, and why you
can often delete a middle layer and get graceful degradation rather than collapse.

**LayerNorm** normalises each token's feature vector to mean 0, variance 1. Note
it is per-token and per-example: it never mixes across the batch or the sequence,
which is exactly why it works with variable-length inputs and batch size 1.

`pre_norm` (normalise *before* the sub-layer, as above) versus `post_norm` (the
original 2017 order) is not cosmetic — see below.

In [ ]:
from learnai.nn.blocks import TransformerBlock

block = TransformerBlock(d_model=8, n_heads=2, causal=True)
block.eval()
out_b = block(x)

# LayerNorm really is per token: statistics are over the final feature axis only.
x_norm = block.ln_1(x)
assert torch.allclose(x_norm.mean(-1), torch.zeros(B, T), atol=1e-5)
assert torch.allclose(x_norm.var(-1, unbiased=False), torch.ones(B, T), atol=2e-4)

# Expand the block's two pre-norm residual updates and verify exact equivalence.
after_attn = x + block.attn(block.ln_1(x))
manual_block = after_attn + block.mlp(block.ln_2(after_attn))
assert torch.allclose(out_b, manual_block, atol=1e-6)
print(f"block: {tuple(x.shape)} -> {tuple(out_b.shape)}   (shape is preserved — that's what lets you stack)")
print(f"LayerNorm mean range: [{x_norm.mean(-1).min():.2e}, {x_norm.mean(-1).max():.2e}]")
print("manual residual + LayerNorm + attention + FFN matches TransformerBlock ✓")

# The residual stream should be recognisable in the output: the block ADDS to it.
cos = torch.nn.functional.cosine_similarity(x.flatten(), out_b.flatten(), dim=0)
print(f"cosine(input, output) = {cos:.3f}  <- an untrained block barely perturbs the stream")

In [ ]:
# Why pre-norm won: gradient magnitude reaching layer 0 of a deep stack.
def grad_at_layer0(pre_norm, n_layers=24):
    torch.manual_seed(0)
    layers = nn.ModuleList([
        TransformerBlock(d_model=32, n_heads=4, causal=True, pre_norm=pre_norm)
        for _ in range(n_layers)
    ])
    h = torch.randn(1, 16, 32, requires_grad=True)
    z = h
    for layer in layers:
        z = layer(z)
    # A random linear readout avoids degenerate probes such as z.sum(): the
    # final post-norm output has zero feature-sum by construction.
    torch.manual_seed(1)
    readout = torch.randn_like(z)
    (z * readout).sum().backward()
    return float(h.grad.norm())

pre_grad = grad_at_layer0(True)
post_grad = grad_at_layer0(False)
assert pre_grad > post_grad
print(f"pre-norm,  24 layers: grad norm at input = {pre_grad:.4f}")
print(f"post-norm, 24 layers: grad norm at input = {post_grad:.4f}")

With post-norm the residual path passes through a LayerNorm at every layer, so the
signal is rescaled 24 times over and the gradient arriving at the input is far
weaker. The original post-norm Transformer relied on careful learning-rate warmup
for stable training; GPT-2 and most later language models moved the norm before
the sub-layer (often replacing LayerNorm with RMSNorm).

---
## 14. Step 13 — Position

`softmax(QKᵀ)V` without a causal mask is permutation-equivariant: shuffle the
input tokens and the outputs merely shuffle in the same way. The mechanism sees
the same set of content vectors in `"dog bites man"` and `"man bites dog"`, with
no stable identity for first, second or third slot. Let us prove it, then fix it.

In [ ]:
perm = torch.tensor([2, 0, 1, 5, 3, 4])

# Use NON-causal attention here: a causal triangle itself encodes an order.
mha_no_pos = MultiHeadAttention(d_model=8, n_heads=2, causal=False)
mha_no_pos.eval()
y_orig = mha_no_pos(x)
y_perm = mha_no_pos(x[:, perm, :])

perm_equivariant = torch.allclose(y_orig[:, perm, :], y_perm, atol=1e-5)
assert perm_equivariant
print("shuffling the input just shuffles the output:", perm_equivariant)
print("  -> attention on its own has NO notion of order")

In [ ]:
from learnai.nn.pos_encoding import SinusoidalPE

pe = SinusoidalPE(d_model=8)
x_pos = pe(x)
position_vectors = pe.pe[0, :T]
assert torch.allclose(pe(torch.zeros_like(x))[0], position_vectors)
print("first three position vectors:")
print(position_vectors[:3])

y_orig = mha_no_pos(x_pos)
y_perm = mha_no_pos(pe(x[:, perm, :]))     # re-encode: position is assigned by slot
position_breaks_symmetry = not torch.allclose(y_orig[:, perm, :], y_perm, atol=1e-5)
assert position_breaks_symmetry
print("with position added, shuffling changes the answer:", position_breaks_symmetry)

In [ ]:
from learnai.viz import plot_pe_matrix

fig, ax = plt.subplots(figsize=(7, 3.4))
plot_pe_matrix(SinusoidalPE(d_model=64).pe[0][:60], ax=ax,
               title="sinusoidal position encoding — fast frequencies left, slow right")
plt.show()

The interactive page **`interactive/positional/index.html`** compares sinusoidal,
learned, RoPE and ALiBi side by side and measures which are genuinely *relative*.
Worth ten minutes with the sliders.

---
## 15. Step 14 — Stack it: that is a GPT

Assemble the pieces:

```
ids ──embed──┐
             ├──> residual stream ──[block] × N──> LayerNorm ──unembed──> logits
position ────┘
```

Everything below is code you have already built in this notebook.

In [ ]:
from learnai.nn import GPT, GPTConfig


class MiniGPT(nn.Module):
    """Every line here was derived above. Compare against learnai/nn/gpt.py."""

    def __init__(self, vocab_size, block_size, n_layer=4, n_head=4, d_model=128, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, d_model)      # step 2
        self.pos_emb = nn.Embedding(block_size, d_model)        # step 13
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([                            # steps 8-12
            TransformerBlock(d_model, n_head, dropout=dropout, causal=True, pre_norm=True)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight             # weight tying
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        # Small logits make a fresh model nearly uniform instead of arbitrarily confident.
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"sequence length {T} exceeds block_size {self.block_size}")
        pos = torch.arange(T, device=idx.device)
        h = self.drop(self.token_emb(idx) + self.pos_emb(pos))  # the stream is born
        for blk in self.blocks:
            h = blk(h)                                          # each block adds to it
        logits = self.lm_head(self.ln_f(h))
        loss = None
        if targets is not None:
            loss = torch.nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Predict one next-token distribution, sample, append, and repeat."""
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]                # crop old context
            logits, _ = self(idx_cond)
            next_logits = logits[:, -1, :] / max(temperature, 1e-8)
            if top_k is not None:
                k = min(top_k, next_logits.size(-1))
                cutoff = next_logits.topk(k).values[:, [-1]]
                next_logits = next_logits.masked_fill(next_logits < cutoff, float("-inf"))
            probs = torch.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

    def num_params(self):
        return sum(p.numel() for p in self.parameters())


m = MiniGPT(vocab_size=65, block_size=64)
print(f"MiniGPT parameters: {m.num_params():,}")

idx = torch.randint(0, 65, (2, 16))
inputs, targets = idx[:, :-1], idx[:, 1:]  # shift by one: x[t] predicts x[t+1]
logits, loss = m(inputs, targets=targets)

# Cross-entropy is the mean negative log-probability assigned to the true next token.
target_logp = logits.log_softmax(-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
manual_loss = -target_logp.mean()
assert torch.allclose(loss, manual_loss, atol=1e-6)

# Load these exact weights into the reusable implementation: assembly must agree.
m_ref = GPT(GPTConfig(vocab_size=65, block_size=64, n_layer=4, n_head=4, d_model=128))
m_ref.load_state_dict(m.state_dict())
ref_logits, ref_loss = m_ref(inputs, targets=targets)
assert torch.allclose(logits, ref_logits, atol=1e-6)
assert torch.allclose(loss, ref_loss, atol=1e-6)
assert abs(loss.item() - math.log(65)) < 0.5
print(f"logits {tuple(logits.shape)}   loss {loss.item():.3f}")
print(f"uniform-guess baseline ln(65) = {math.log(65):.3f}")
print("manual NLL and learnai.nn.GPT both match MiniGPT ✓")

That loss check is the single most useful sanity test in language modelling. A
freshly initialised model should be close to guessing uniformly, so the loss
should sit near `ln(vocab_size)`. If it starts much lower, something is leaking the
answer (a shifted target, a missing causal mask). If much higher, the
initialisation is broken.

---
## 16. Step 15 — Loss and training on real text

~1MB of Shakespeare, character level, 1,500 steps. On an M-series laptop this
takes about a minute. For a quick executable smoke test, launch Jupyter with
`LEARNAI_FAST=1`; it keeps every cell and assertion but uses 12 training steps
and a shorter sample. Leave it unset for the meaningful learning run.

Watch what the samples do as loss falls: first spaces appear, then word-shaped
blobs, then real words, then `SPEAKER:` lines, then something with the rhythm of
dialogue. You are watching a curriculum the model chose for itself.

In [ ]:
import functools

from learnai.data import CharTokenizer, get_batch, tiny_shakespeare, train_val_split
from learnai.train import TrainConfig, Trainer

text = tiny_shakespeare()
tok = CharTokenizer(text)
train_ids, val_ids = train_val_split(tok.encode(text))
print(f"{len(text):,} chars, vocab {tok.vocab_size}, "
      f"{len(train_ids):,} train / {len(val_ids):,} val tokens")

block_size, batch_size = ((64, 4) if FAST else (128, 32))
d_model_train = 64 if FAST else 128
model = MiniGPT(vocab_size=tok.vocab_size, block_size=block_size,
                n_layer=4, n_head=4, d_model=d_model_train, dropout=0.1)
print(f"model parameters: {model.num_params():,}")

In [ ]:
train_steps = 12 if FAST else 1500
trainer = Trainer(model, TrainConfig(
    steps=train_steps, lr=3e-3, warmup_steps=(2 if FAST else 100),
    eval_every=(6 if FAST else 250), eval_steps=(2 if FAST else 20),
    log_every=(6 if FAST else 250),
))
history = trainer.fit(
    functools.partial(get_batch, train_ids, batch_size, block_size),
    functools.partial(get_batch, val_ids, batch_size, block_size),
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
history.plot(ax=axes[0])
axes[0].set_title("loss")
axes[1].plot(history.steps, history.lrs, color="#eb6834")
axes[1].set_title("learning rate (warmup, then cosine decay)")
axes[1].set_xlabel("step")
axes[1].set_ylabel("lr")
plt.tight_layout()
plt.show()

initial, final = history.val_loss[0], history.val_loss[-1]
if not FAST:
    assert final < initial
trend = "training improved it ✓" if final < initial else "smoke run is too short for a trend"
print(f"validation loss: {initial:.3f} -> {final:.3f}  ({trend})")
print(f"final perplexity {math.exp(final):.1f}")
print(f"a uniform model would score {math.log(tok.vocab_size):.3f}  ->  perplexity {tok.vocab_size}")

In [ ]:
prompt_text = "\n"
prompt = torch.tensor([tok.encode(prompt_text)], dtype=torch.long, device=trainer.device)
sample_length = 40 if FAST else 400
sample = model.generate(prompt, max_new_tokens=sample_length, temperature=0.8, top_k=20)
print(tok.decode(sample[0].tolist()))

---
## 17. Step 16 — Look inside the model you just trained

The most valuable habit in this whole field: after training, **open the model and
look**. `learnai.interp.capture_attention` flips a flag on every attention module
and collects the maps from one forward pass.

In [ ]:
from learnai.interp import capture_attention
from learnai.viz import attention_grid

probe_text = "First Citizen:\nBefore we proceed"
probe = torch.tensor([tok.encode(probe_text)], device=trainer.device)
probe_tokens = [c.replace("\n", "\\n") for c in probe_text]

model.eval()
with capture_attention(model) as attn:
    model(probe)

for name, a in attn.items():
    print(f"{name:20s} {tuple(a.shape)}")

In [ ]:
attention_grid(attn["blocks.0.attn"][0].cpu(), tokens=probe_tokens,
               title="layer 0 — all heads")
plt.show()

attention_grid(attn["blocks.3.attn"][0].cpu(), tokens=probe_tokens,
               title="layer 3 — all heads")
plt.show()

Patterns to hunt for, roughly in order of how often they show up:

- **diagonal** — attends to itself; the head is mostly passing content through
- **sub-diagonal** — a *previous-token head*; one of the two components of an
  induction circuit
- **bright first column** — an *attention sink*: the head has nothing to do on this
  input and parks its mass on token 0. Extremely common, and the reason naive
  KV-cache eviction of the first token wrecks a model
- **vertical stripe** — everyone reads one specific token, usually a delimiter

Early layers tend toward positional patterns, later layers toward content.

In [ ]:
from learnai.nn.attention import attention_entropy

print(f"{'layer':<10}{'head':<6}{'mean entropy':>14}{'  behaviour'}")
for name, a in attn.items():
    H = attention_entropy(a)[0].mean(dim=-1)     # (n_heads,)
    for h, e in enumerate(H):
        n = a.shape[-1]
        note = ("focused — locked onto ~1 position" if e < 0.7
                else "diffuse — averaging broadly" if e > math.log(n) * 0.75
                else "selective")
        print(f"{name.split('.')[1]:<10}{h:<6}{e:>14.3f}  {note}")

> **Your turn.** Ablate a head and measure what it was worth. Use `learnai.interp.ablate(model, 'blocks.3.attn', mode='mean')` and compare validation loss inside and outside the context manager. Which layer hurts most when you remove it?
>
> <details><summary>hint</summary>
>
> `trainer.evaluate(functools.partial(get_batch, val_ids, batch_size, block_size))` gives you a loss number in one call.
>
> </details>

In [ ]:
# from learnai.interp import ablate  # uncomment when you implement the loop below
val_fn = functools.partial(get_batch, val_ids, batch_size, block_size)
baseline = trainer.evaluate(val_fn)
print(f'baseline {baseline:.4f}')

# for each block, ablate its attention and re-measure

> **Your turn.** Train a **1-layer** model and a **2-layer** model on the induction task from `learnai.data.induction_task`, and measure accuracy *only at the position that requires the circuit*. A one-layer model provably cannot solve it — an induction head needs two heads composing across layers. Confirm the gap.
>
> <details><summary>hint</summary>
>
> `induction_task` returns `(x, y, positions)`; score `logits[range(B), positions]` against `y[range(B), positions]`.
>
> </details>

In [ ]:
# your code here

---
## 18. Where to go next

You have built every component of a GPT and trained one. What is left is scale and
engineering, not new concepts.

**In this repo**

| | |
|---|---|
| `interactive/attention/` | grab the mechanism and move it — the √d effect is visceral there |
| `interactive/positional/` | RoPE and ALiBi, measured rather than asserted |
| `learnai/nn/gpt.py` | the reference GPT, ~200 annotated lines |
| `learnai/interp/hooks.py` | `ablate` and `patch` — the tools for causal claims |

**Concepts deliberately left out, in the order they usually bite**

1. **KV caching** — `generate` here re-runs the whole prefix every step, O(T²) per
   token. Caching K and V makes it O(T). This is the single biggest inference win.
2. **Flash attention** — never materialise the `(T, T)` matrix; tile it through SRAM.
   Same math, ~4× faster, and it is why context windows grew.
3. **RoPE and RMSNorm** — the modern replacements for learned PE and LayerNorm.
   Both are in `learnai/nn/`; swapping them into `GPT` is a genuinely useful exercise.
4. **Mixture of Experts** — replace the MLP with N MLPs and route each token to 2.
   Parameters grow, FLOPs do not.
5. **RLHF / DPO** — everything above produces a *text predictor*. Turning one into
   an assistant is a separate training stage with a separate objective.

**Papers worth reading in this order**

- *Attention Is All You Need* (2017) — you have now implemented it; the paper reads
  very differently on the other side of that
- *Language Models are Unsupervised Multitask Learners* (GPT-2, 2019) — scale as a
  method
- *In-context Learning and Induction Heads* (2022) — the first fully
  reverse-engineered circuit, and the best argument that these models are
  understandable
- *Training Compute-Optimal Large Language Models* (Chinchilla, 2022) — why model
  size alone was the wrong axis